In [12]:
import math
from scipy.special import ellipk, ellipe

In [17]:
def CalcSelfInductance(h_mm: float, w_mm: float, r_avg_mm: float, rho_c: float, f: float) -> float:
    h = h_mm / 1000.0 # Convert to meters
    w = w_mm / 1000.0 # Convert to meters
    r_avg = r_avg_mm / 1000.0 # Convert to meters

    mu_0 = 4 * math.pi * 1e-7 # Permeability of free space (H/m)

    mu_r = 1.0
    sigma = 1.0 / rho_c # Conductivity of copper (S/m)
    GMD = math.exp(0.5 * math.log(h * h + w * w) + 2 * w / (3 * h) * math.atan(h / w) + 2 * h / (3 * w) * math.atan(w / h) - w * w / (12 * h * h) * math.log(1 + h * h / (w * w)) - h * h / (12 * w * w) * math.log(1 + w * w / (h * h)) - 25 / 12)
    print(f"r_avg: {r_avg} GMD: {GMD}")
    # Internal inductance
    #L_int_low = mu_0 * mu_r / (8 * math.pi)  # Low-frequency internal inductance
    #print(f"L_int_low: {L_int_low / 1e-6} uH")
    omega = 2 * math.pi * f      # Angular frequency
    delta = math.sqrt(2 / (omega * mu_0 * mu_r * sigma))  # Skin depth
    #L_int = L_int_low * min(1, delta / (GMD / 2))  # Smooth transition
    L_int = 0.0
    L_s = L_int + mu_0 * r_avg * (math.log(8 * r_avg / GMD) - 2)
    #Console.WriteLine(f"r_avg: {r_avg} GMD: {GMD} L_s: {L_s / 1e-9} L_s/l: {L_s / (2 * Math.PI * r_avg) / 1e-9}")
    return L_s

In [18]:
def CalcInductanceCoaxLoops(r_a_mm: float, z_a_mm: float, r_b_mm: float, z_b_mm: float) -> float:
    r_a = r_a_mm / 1000.0 # Convert to meters
    z_a = z_a_mm / 1000.0 # Convert to meters
    r_b = r_b_mm / 1000.0 # Convert to meters
    z_b = z_b_mm / 1000.0 # Convert to meters

    mu_0 = 4 * math.pi * 1e-7 # Permeability of free space (H/m)

    if r_a == 0 or r_b == 0:
        return 0

    d = abs(z_b - z_a)
    k = math.sqrt(4 * r_a * r_b / ((r_a + r_b) * (r_a + r_b) + d * d))
    m = k * k
    L_ab = 2 * mu_0 / k * math.sqrt(r_a * r_b) * ((1 - k * k / 2) * ellipk(m) - ellipe(m))
    return L_ab

# r is mean radius of each turn
# z is height of mid-point
def CalcLyleLoops(r: float, z: float, w: float, h: float) -> tuple[float, float, float, float]:
    r_i: float
    z_i: float
    r_j: float
    z_j: float

    if h > w:
        # Two rings with r_avg = r_1 and at d +/- beta
        r_adj = r * (1 + w * w / (24 * r * r))
        beta = math.sqrt(((h * h) - (w * w)) / 12)
        r_i = r_adj
        z_i = z + beta
        r_j = r_adj
        z_j = z - beta
    else:
        # Two rings with r_avg = r_1 and r_1 + 2*delta and at z = d
        r_adj = r * (1 + h * h / (24 * r * r))
        delta = math.sqrt(((w * w) - (h * h)) / 12)
        r_i = r_adj - delta
        z_i = z
        r_j = r_adj + delta
        z_j = z

    return (r_i, z_i, r_j, z_j)

def CalcMutualInductance_Lyle(r_a: float, z_a: float, h_a: float, w_a: float, r_b: float, z_b: float, h_b: float, w_b: float) -> float:
    ta = CalcLyleLoops(r_a, z_a, w_a, h_a)
    r_1, z_1, r_2, z_2 = ta

    tb = CalcLyleLoops(r_b, z_b, w_b, h_b)
    r_3, z_3, r_4, z_4 = tb

    L_13 = CalcInductanceCoaxLoops(r_1, z_1, r_3, z_3)
    L_14 = CalcInductanceCoaxLoops(r_1, z_1, r_4, z_4)
    L_23 = CalcInductanceCoaxLoops(r_2, z_2, r_3, z_3)
    L_24 = CalcInductanceCoaxLoops(r_2, z_2, r_4, z_4)
    L_ab = (L_13 + L_14 + L_23 + L_24) / 4

    return L_ab

In [19]:
def convert_units(value: float, from_unit: str, to_unit: str) -> float:
	conversions = {
		('in', 'mm'): 25.4,
		('mm', 'in'): 1 / 25.4,
	}

	if from_unit == to_unit:
		return value

	try:
		return value * conversions[(from_unit, to_unit)]
	except KeyError as error:
		raise ValueError(f"Unsupported conversion: {from_unit} to {to_unit}") from error


#StrandHeight_mm = Conversions.in_to_mm(0.3),
#StrandWidth_mm = Conversions.in_to_mm(0.085),
#CornerRadius_mm = Conversions.in_to_mm(0.032),
#InsulationThickness_mm = Conversions.in_to_mm(0.018),
t_ins = convert_units(0.018, 'in', 'mm')
h_a = convert_units(0.3, 'in', 'mm')
w_a = convert_units(0.085, 'in', 'mm')
z_a = 0.0
r_a = convert_units(15.25, 'in', 'mm') + (2*t_ins + w_a) / 2.0
h_b = convert_units(0.3, 'in', 'mm')
w_b = convert_units(0.085, 'in', 'mm')
z_b = 0.0
r_b = convert_units(25.25, 'in', 'mm') + (2*t_ins + w_b) / 2.0

L_a = CalcSelfInductance(h_a, w_a, r_a, 1.68e-8, 0.01)
L_b = CalcSelfInductance(h_b, w_b, r_b, 1.68e-8, 0.01)
L_ab = CalcMutualInductance_Lyle(r_a, z_a, h_a, w_a, r_b, z_b, h_b, w_b)
L_ba = CalcMutualInductance_Lyle(r_b, z_b, h_b, w_b, r_a, z_a, h_a, w_a)

print (f"L_a: {L_a / 1e-6:.3f} uH")
print (f"L_b: {L_b / 1e-6:.3f} uH")
print (f"L_ab: {L_ab / 1e-6:.3f} uH")
print (f"L_ba: {L_ba / 1e-6:.3f} uH")

r_avg: 0.3888867 GMD: 0.002187368329525813
r_avg: 0.6428866999999999 GMD: 0.002187368329525813
L_a: 2.571 uH
L_b: 4.656 uH
L_ab: 0.548 uH
L_ba: 0.548 uH


In [20]:
from inductance.self import L_lyle6
from inductance.mutual import mutual_lyles_method, mutual_rayleigh

# L_lyle6(r, dr, dz, n) expects meters: dr = radial width, dz = axial height
L_a_package = L_lyle6(r_a / 1000.0, w_a / 1000.0, h_a / 1000.0, 1)
L_b_package = L_lyle6(r_b / 1000.0, w_b / 1000.0, h_b / 1000.0, 1)
L_ab_lyles = mutual_lyles_method(r_a / 1000.0, 0.0, w_a / 1000.0, h_a / 1000.0, 1, r_b / 1000.0, 0.0, w_b / 1000.0, h_b / 1000.0, 1)
L_ab_rayleigh = mutual_rayleigh(r_a / 1000.0, 0.0, w_a / 1000.0, h_a / 1000.0, 1, r_b / 1000.0, 0.0, w_b / 1000.0, h_b / 1000.0, 1)
print(f"L_a_package: {L_a_package / 1e-6:.3f} uH")
print(f"L_b_package: {L_b_package / 1e-6:.3f} uH")
print(f"L_ab_lyles: {L_ab_lyles / 1e-6:.3f} uH")
print(f"L_ab_rayleigh: {L_ab_rayleigh / 1e-6:.3f} uH")

L_a_package: 2.571 uH
L_b_package: 4.656 uH
L_ab_lyles: 0.548 uH
L_ab_rayleigh: 0.548 uH
